In [ ]:
# %% [markdown]
# # 제품별 페르소나 구축 노트북
# 
# **입력 파일:**
# - `product.json`: 제품 정보 목록
# - `people_segment.json`: 소비자 세그먼트 및 개인 정보 목록
# 
# **출력 파일:**
# - `/mnt/data/product_personas.json`: 모든 제품에 대한 페르소나 정보를 통합한 JSON 파일
# - `/mnt/data/personas/<product>.json`: 제품별로 개별 페르소나 JSON 파일
# - `/mnt/data/personas/manifest.json`: 개별 파일 목록
# - `/mnt/data/product_personas_split.zip`: 개별 페"르소나 JSON 파일들을 압축한 ZIP 파일
# 
# **설명:**
# 이 노트북은 제품 정보와 소비자 세그먼트 정보를 결합하여 각 제품에 가장 적합한 소비자 페르소나를 생성합니다. '적합도(fit score)'를 계산하여 상위 페르소나를 선정하고, 해당 페르소나의 인구통계학적 정보, 쇼핑 프로필, 성향 지표 등을 바탕으로 상세한 속성(attributes) 가중치를 계산합니다.

# %% [markdown]
# ## 1. 라이브러리 임포트 및 기본 설정
# 
# 필요한 라이브러리를 임포트합니다.

# %%
import json
import re
import argparse
from pathlib import Path
from typing import List, Dict, Any

import pandas as pd
import zipfile

# %% [markdown]
# ## 2. 헬퍼 함수 및 정규화 로직
# 
# 데이터 정규화 및 전처리를 위한 헬퍼 함수들을 정의합니다.

# %%
# 텍스트 정규화 (공백 제거, 소문자 변환)
def norm_text(s: Any) -> str:
    return re.sub(r"\s+", "", str(s)).strip().lower()

# 가격대를 'low', 'mid', 'high'로 버킷팅
def price_bucket(price: float) -> str:
    if price is None:
        return "mid"
    try:
        p = float(price)
    except Exception:
        p = 0.0
    if p >= 10000:
        return "high"
    if p <= 3000:
        return "low"
    return "mid"

# 구매 주기를 월별 구매 횟수로 변환
def purchase_cycle_to_monthly_base(cycle: str) -> int:
    if not cycle:
        return 1
    c = norm_text(cycle)
    if "주4~6회" in c or "주4-6회" in c:
        return 20
    if "주2~3회" in c or "주2-3회" in c:
        return 10
    if "주1회" in c:
        return 4
    if "2주일에1회" in c or "격주" in c:
        return 2
    if "1달에1회" in c or "월1회" in c:
        return 1
    return 1

# 카테고리 매핑 정보
CATEGORY_MAP = {
    "요거트": ["유가공품", "발효유", "우유", "요구르트"],
    "커피음료": ["커피및차", "커피", "커피음료"],
    "참치캔": ["조미수산가공품", "수산물통조림", "염건수산가공품"],
    "축산캔": ["축산캔", "육류가공품"],
    "참치액/조미료": ["조미식품", "장류", "조미소스", "조미료", "소스"],
    "유지류": ["유지류", "참기름", "들기름", "식용유"],
}

# 제품 정보로부터 제품 그룹 추론
def infer_product_group(p: Dict[str, Any]) -> str:
    name = p.get("product_name", "")
    cat1 = p.get("category", {}).get("level_1", "")
    cat2 = p.get("category", {}).get("level_2", "")
    cat3 = p.get("category", {}).get("level_3", "")
    c = norm_text(f"{cat1} {cat2} {cat3}")
    if "요거트" in name or "하이그릭" in name:
        return "요거트"
    if "라떼" in name or "카페라떼" in name or "바닐라" in name:
        return "커피음료"
    if "리챔" in name or "오믈레햄" in name or "햄" in name:
        return "축산캔"
    if "참치액" in name:
        return "참치액/조미료"
    if "참기름" in name:
        return "유지류"
    if "참치" in name or "캔" in name or "라이트스탠다드참치" in c:
        return "참치캔"
    if "조미료" in p.get("category", {}).get("level_2", ""):
        return "참치액/조미료"
    return "기타"

# 구매 결정 요인(Decision Criteria) 정규화
DC_NORMALIZE = {
    "맛": ["맛"],
    "가격": ["가격"],
    "품질": ["품질"],
    "안전성": ["안전성", "영양(건강)", "영양"],
    "조리의 편리성": ["조리의 편리성"],
    "구입의 편리성": ["구입의 편리성"],
    "신선도": ["신선도 (제조일자, 소비(유통)기한 포함)", "신선도"],
}

def normalize_dc_list(dc_list: Any) -> List[str]:
    if not dc_list:
        return []
    base_set = set()
    for base, syns in DC_NORMALIZE.items():
        for item in dc_list:
            if not item:
                continue
            item_clean = str(item).replace(" ", "")
            for syn in syns:
                if item_clean.replace(" ", "").startswith(syn.replace(" ", "")):
                    base_set.add(base)
    return list(base_set)

# 제품 특징과 페르소나 성향 지표 매핑
FEATURE_TO_ORI = {
    "건강식품": "health",
    "고단백": "health",
    "프리미엄": "premium",
    "간편함": "convenience",
    "가성비": "price_sensitivity",
    "매콤한맛": "variety_seeking",
    "고소한맛": "taste",
    "감칠맛": "taste",
    "높은 만족도": None,
}


# %% [markdown]
# ## 3. 핵심 로직: 적합도 계산 및 속성 구축
# 
# 제품과 소비자 페르소나 간의 적합도를 계산하고, 이를 바탕으로 페르소나의 주요 속성과 가중치를 생성합니다.

# %%
def compute_fit(product: Dict[str, Any], person: Dict[str, Any]) -> float:
    """제품과 소비자 간의 적합도 점수를 계산합니다."""
    score = 0.0

    # 1) 카테고리 선호도 (상위 1, 2위 카테고리에 가중치 부여)
    pg = infer_product_group(product)
    cat_keywords = [norm_text(k) for k in CATEGORY_MAP.get(pg, [])]
    top_cats = [norm_text(x) for x in (person.get("shopping_profile", {}).get("top_categories") or []) if x and x != "응답없음"]
    if top_cats:
        if len(top_cats) >= 1 and any(k in top_cats[0] for k in cat_keywords):
            score += 1.0
        if len(top_cats) >= 2 and any(k in top_cats[1] for k in cat_keywords):
            score += 0.5

    # 2) 가구 형태 매칭 (제품 타겟과 가구 형태 일치 여부)
    tgt = product.get("targeted_consumer", "") or ""
    if tgt:
        tgt_tokens = [t.strip() for t in tgt.split(",")]
        hh = person.get("demographics", {}).get("household", "") or ""
        for t in tgt_tokens:
            if t and t in hh:
                score += 1.0
                break

    # 3) 제품 특징 - 소비자 성향 매칭
    feats = [f.strip() for f in (product.get("feature", "") or "").split(",")]
    ori = person.get("orientations", {}) or {}
    for f in feats:
        key = FEATURE_TO_ORI.get(f.strip(), None)
        if key in ori and isinstance(ori[key], (int, float)):
            score += ori[key] / 9.0  # 1~9점 척도를 0~1로 정규화
        elif key == "taste":
            score += 0.2

    # 4) 구매 결정 요인 일치도
    dc = normalize_dc_list(person.get("shopping_profile", {}).get("decision_criteria"))
    for base in dc:
        if base in ["맛", "가격", "품질", "안전성"]:
            score += 0.2
        if base in ["조리의 편리성", "구입의 편리성"]:
            score += 0.15
        if base == "신선도" and pg in ["축산캔", "참치캔", "요거트", "커피음료"]:
            score += 0.1

    # 5) 가격 적합도
    pb = price_bucket(product.get("price", 0))
    ps = (ori.get("price_sensitivity", 5) or 5) / 9.0
    prem = (ori.get("premium", 5) or 5) / 9.0
    if pb == "low":
        score += 0.5 * ps + 0.1
    elif pb == "high":
        score += 0.6 * prem
    else:
        score += 0.2 * (prem + ps)

    return float(score)

def build_attributes(product: Dict[str, Any], person: Dict[str, Any], min_attrs: int = 10, cap_attrs: int = 14) -> List[Dict[str, Any]]:
    """제품과 페르소나를 기반으로 속성(attribute)과 가중치를 생성합니다."""
    ori = person.get("orientations", {}) or {}
    dc = normalize_dc_list(person.get("shopping_profile", {}).get("decision_criteria"))
    feats = [f.strip() for f in (product.get("feature", "") or "").split(",")]
    name = product.get("product_name", "")
    pg = infer_product_group(product)

    extra_attrs: List[str] = []
    # 제품별 특화 속성 추가
    if "요거트" in name or pg == "요거트":
        if "유당불내증" in (product.get("targeted_consumer") or ""):
            extra_attrs.append("락토프리/소화")
        if "고단백" in feats:
            extra_attrs.append("고단백")
    if "참치액" in name:
        extra_attrs.append("감칠맛")
        if "500g" in name: extra_attrs.append("용량_500g")
        if "900g" in name: extra_attrs.append("용량_900g")
        if "진" in name:   extra_attrs.append("진한맛")
        if "순" in name:   extra_attrs.append("깔끔한맛")
        if "프리미엄" in name: extra_attrs.append("프리미엄원재료")
    if "참기름" in name:
        extra_attrs.append("고소한맛")
        if "90g" in name: extra_attrs.append("소용량_90g")
        if "135g" in name: extra_attrs.append("중간용량_135g")
        if "매콤" in name: extra_attrs.append("매콤풍미")
    if "리챔" in name:
        extra_attrs.append("간편조리")
        extra_attrs.append("낮은칼로리")
        if "200g" in name: extra_attrs.append("소용량_200g")
        if "340g" in name: extra_attrs.append("대용량_340g")
    if "라떼" in name or "카페라떼" in name:
        extra_attrs.append("간편섭취")
        if "바닐라" in name: extra_attrs.append("단맛취향")
        if "유당불내증" in (product.get("targeted_consumer") or ""):
            extra_attrs.append("락토프리/소화")

    # 기본 속성 목록
    attrs = [
        "확인_맛","확인_가격","확인_브랜드","확인_품질","확인_안전성","확인_용량",
        "편의성_조리","편의성_구입","프리미엄지향","건강/영양",
        "가격민감","브랜드충성","다양성추구"
    ] + extra_attrs

    w = {a: 0.0 for a in attrs}

    # 1. 소비자 성향 지표 기반 가중치 부여
    w["건강/영양"]    += (ori.get("health", 5) or 5) / 9.0
    w["가격민감"]     += (ori.get("price_sensitivity", 5) or 5) / 9.0
    w["편의성_조리"]  += (ori.get("hmr", 5) or 5) / 9.0
    w["브랜드충성"]   += (ori.get("brand_loyalty", 5) or 5) / 9.0
    w["프리미엄지향"]  += (ori.get("premium", 5) or 5) / 9.0
    w["편의성_구입"]   += (ori.get("convenience", 5) or 5) / 9.0
    w["다양성추구"]   += (ori.get("variety_seeking", 5) or 5) / 9.0

    # 2. 구매 결정 요인 기반 가중치 추가
    for base in dc:
        if base == "맛": w["확인_맛"] += 1.0
        if base == "가격": w["확인_가격"] += 1.0
        if base == "품질": w["확인_품질"] += 1.0
        if base == "안전성": w["확인_안전성"] += 1.0
        if base == "조리의 편리성": w["편의성_조리"] += 0.8
        if base == "구입의 편리성": w["편의성_구입"] += 0.8
        if base == "신선도":
            w["확인_품질"] += 0.5
            w["확인_안전성"] += 0.5

    # 3. 제품 특징 기반 가중치 추가
    for f in feats:
        f = f.strip()
        if f in ["고단백","건강식품"] and "건강/영양" in w: w["건강/영양"] += 0.6
        if f in ["프리미엄"] and "프리미엄지향" in w: w["프리미엄지향"] += 0.6
        if f in ["간편함"]:
            w["편의성_구입"] += 0.4
            w["편의성_조리"] += 0.4
        if f in ["가성비"]:
            w["확인_가격"] += 0.6
            w["가격민감"] += 0.4
        if f in ["매콤한맛","고소한맛","감칠맛"] and "확인_맛" in w: w["확인_맛"] += 0.4

    # 4. 특화 속성 가중치 강조
    for a in extra_attrs:
        w[a] = w.get(a, 0.0) + 0.8

    # 5. 가중치 정규화 및 상위 K개 선택
    w_series = pd.Series(w)
    w_series = w_series[w_series > 0]
    if len(w_series) < min_attrs:
        need = min_attrs - len(w_series)
        zeros = pd.Series({k: 0.0001 for k in w if k not in w_series})
        w_series = pd.concat([w_series, zeros.iloc[:need]])
    
    w_series = (w_series / w_series.sum()).sort_values(ascending=False)
    w_series = w_series.iloc[:cap_attrs]
    
    return [{"name": k, "weight": round(float(v), 4)} for k, v in w_series.items()]


# %% [markdown]
# ## 4. 페르소나 생성 및 파일 출력
# 
# 위에서 정의한 함수들을 사용하여 최종 페르소나를 생성하고, 다양한 형식의 파일로 저장합니다.

# %%
def slugify(name: str) -> str:
    """파일 이름으로 사용하기 안전한 문자열로 변환합니다."""
    s = re.sub(r"\s+", "_", name.strip())
    s = re.sub(r"[^0-9A-Za-z가-힣_]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "product"

def build_personas(
    products: List[Dict[str, Any]],
    people: List[Dict[str, Any]],
    topk: int = 3
) -> Dict[str, Any]:
    """모든 제품에 대해 상위 K개의 페르소나를 구축합니다."""
    out = {}
    for p in products:
        scored = []
        for person in people:
            s = compute_fit(p, person)
            scored.append((person, s))
        scored.sort(key=lambda x: x[1], reverse=True)

        # 다양성 확보를 위해 segment_id가 중복되지 않도록 선정
        selected = []
        used_segments = set()
        for person, s in scored:
            seg = person.get("segment_id")
            if seg in used_segments:
                continue
            selected.append((person, s))
            used_segments.add(seg)
            if len(selected) >= topk:
                break

        persona_list = []
        for person, s in selected:
            persona_list.append({
                "product_name": p.get("product_name"),
                "product_group": infer_product_group(p),
                "price": p.get("price"),
                "targeted_consumer": p.get("targeted_consumer"),
                "segment_id": person.get("segment_id"),
                "segment_label": person.get("segment_label"),
                "source_persona_id": person.get("persona_id"),
                "demographics": person.get("demographics"),
                "shopping_profile": person.get("shopping_profile"),
                "orientations": person.get("orientations"),
                "fit_score": round(float(s), 3),
                "monthly_base_freq": purchase_cycle_to_monthly_base(person.get("shopping_profile",{}).get("purchase_cycle")),
                "attributes": build_attributes(p, person),
            })
        out[p.get("product_name")] = persona_list
    return out

def split_per_product(data: Dict[str, Any], out_dir: Path) -> Path:
    """페르소나 데이터를 제품별 개별 파일로 저장하고 압축합니다."""
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest = []
    files = []
    for product_name, personas in data.items():
        slug = slugify(product_name)
        out_path = out_dir / f"{slug}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump({product_name: personas}, f, ensure_ascii=False, indent=2)
        manifest.append({"product_name": product_name, "file": str(out_path)})
        files.append(out_path)

    manifest_path = out_dir / "manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    # zip 파일로 묶기
    zip_path = out_dir.parent / "product_personas_split.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in files:
            zf.write(p, arcname=p.name)
    return zip_path


# %% [markdown]
# ## 5. 실행 (Main)
# 
# 스크립트의 `main` 함수에 해당하는 부분입니다. 입력/출력 경로를 설정하고, 전체 프로세스를 실행합니다.

# %%
# --- 설정 (argparse 대신 직접 변수 지정) ---
# 입력/출력 데이터가 저장된 기본 경로를 지정합니다.
# 로컬 환경에서 실행할 경우, 이 경로를 실제 파일 위치에 맞게 수정해주세요.
DATA_DIR = Path("/mnt/data")

# 입력 파일 경로
input_products_path = DATA_DIR / "product.json"
input_people_path = DATA_DIR / "people_segment.json"

# 출력 디렉토리 경로
output_dir_path = DATA_DIR

# 제품별로 생성할 상위 페르소나 개수
TOP_K = 3

# --- 실행 로직 ---
# 1. 데이터 로드
print(f"제품 데이터 로딩: {input_products_path}")
with open(input_products_path, "r", encoding="utf-8") as f:
    products_data = json.load(f)

print(f"소비자 데이터 로딩: {input_people_path}")
with open(input_people_path, "r", encoding="utf-8") as f:
    people_data = json.load(f)

# 2. 페르소나 생성
print("\n제품별 페르소나 생성 중...")
product_personas = build_personas(products_data, people_data, topk=TOP_K)

# 3. 통합 JSON 파일 저장
output_json_path = output_dir_path / "product_personas.json"
print(f"통합 페르소나 파일 저장: {output_json_path}")
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(product_personas, f, ensure_ascii=False, indent=2)

# 4. 제품별 파일 분리 및 압축
split_dir_path = output_dir_path / "personas"
print(f"제품별 페르소나 파일 분리 및 압축 중... -> {split_dir_path}")
zip_file_path = split_per_product(product_personas, split_dir_path)

# 5. 최종 결과 요약 출력
total_products = len(product_personas)
total_personas = sum(len(v) for v in product_personas.values())
print("\n--- 작업 완료 ---")
print(f"[OK] 총 제품 수: {total_products}, 총 생성된 페르소나 수: {total_personas}")
print(f"[출력] 통합 JSON 파일: {output_json_path}")
print(f"[출력] 개별 JSON 파일 경로: {split_dir_path}")
print(f"[출력] 압축 ZIP 파일: {zip_file_path}")